In [1]:
# ============================================================
# REPRENDRE LE NOTEBOOK FINAL
# VERSION ROBUSTE POUR LE TEST PUBLIC ET CACHÉ
# ============================================================

from pathlib import Path
import json


INPUT_ROOT = Path("/kaggle/input")


# ------------------------------------------------------------
# 1. RETROUVER LE CHECKPOINT V9
# ------------------------------------------------------------

v9_checkpoint_candidates = []


for path in INPUT_ROOT.rglob(
    "v9_transformer_checkpoint"
):
    if not path.is_dir():
        continue

    required_checkpoint_files = [
        path / "feature_lists.json",
    ]

    if all(
        file_path.exists()
        for file_path in required_checkpoint_files
    ):
        v9_checkpoint_candidates.append(
            path
        )


v9_checkpoint_candidates = sorted(
    set(v9_checkpoint_candidates),
    key=lambda path: (
        "donne-lui-un-nom-comme-llm-arena-final-inference"
        not in str(path).lower(),
        len(str(path)),
        str(path),
    ),
)


if not v9_checkpoint_candidates:
    raise FileNotFoundError(
        "Le dossier v9_transformer_checkpoint "
        "contenant feature_lists.json est introuvable."
    )


V9_DIR = v9_checkpoint_candidates[0]


print("Checkpoint V9 sélectionné :")
print(V9_DIR)


# ------------------------------------------------------------
# 2. RETROUVER LES MODÈLES FINAUX
# ------------------------------------------------------------

required_model_filenames = [
    "lightgbm_seed_2028.joblib",
    "lightgbm_seed_31415.joblib",
    "lightgbm_seed_98765.joblib",
    "logistic_final.joblib",
    "metadata.json",
]


final_model_candidates = []


for path in INPUT_ROOT.rglob(
    "v9_final_models"
):
    if not path.is_dir():
        continue

    if all(
        (path / filename).exists()
        for filename in required_model_filenames
    ):
        final_model_candidates.append(
            path
        )


final_model_candidates = sorted(
    set(final_model_candidates),
    key=lambda path: (
        "donne-lui-un-nom-comme-llm-arena-final-inference"
        not in str(path).lower(),
        len(str(path)),
        str(path),
    ),
)


if not final_model_candidates:
    raise FileNotFoundError(
        "Le dossier v9_final_models contenant "
        "les 3 LightGBM, le modèle logistique "
        "et metadata.json est introuvable."
    )


FINAL_MODEL_DIR = final_model_candidates[0]


print("\nDossier des modèles sélectionné :")
print(FINAL_MODEL_DIR)


# ------------------------------------------------------------
# 3. CHARGER LES MÉTADONNÉES
# ------------------------------------------------------------

metadata_path = (
    FINAL_MODEL_DIR
    / "metadata.json"
)


try:
    with metadata_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        metadata = json.load(file)

except json.JSONDecodeError as error:
    raise ValueError(
        "metadata.json n'est pas un fichier JSON valide."
    ) from error


if not isinstance(metadata, dict):
    raise TypeError(
        "Le contenu de metadata.json "
        "doit être un dictionnaire."
    )


# ------------------------------------------------------------
# 4. VÉRIFIER LES FICHIERS NÉCESSAIRES
# ------------------------------------------------------------

required_v9_paths = [
    V9_DIR / "feature_lists.json",
]


required_model_paths = [
    FINAL_MODEL_DIR / filename
    for filename in required_model_filenames
]


missing_paths = [
    path
    for path in (
        required_v9_paths
        + required_model_paths
    )
    if not path.is_file()
]


if missing_paths:
    raise FileNotFoundError(
        "Fichiers nécessaires manquants :\n"
        + "\n".join(
            str(path)
            for path in missing_paths
        )
    )


# ------------------------------------------------------------
# 5. VÉRIFIER LA CONFIGURATION DU BLEND
# ------------------------------------------------------------

LGB_WEIGHT_FROM_METADATA = float(
    metadata.get(
        "lgb_weight",
        0.94,
    )
)

LOGISTIC_WEIGHT_FROM_METADATA = float(
    metadata.get(
        "logistic_weight",
        0.06,
    )
)

TEMPERATURE_FROM_METADATA = float(
    metadata.get(
        "temperature",
        0.99,
    )
)


if abs(
    (
        LGB_WEIGHT_FROM_METADATA
        + LOGISTIC_WEIGHT_FROM_METADATA
    )
    - 1.0
) > 1e-8:
    raise ValueError(
        "Les poids LightGBM et Logistic "
        "ne totalisent pas 1."
    )


if TEMPERATURE_FROM_METADATA <= 0:
    raise ValueError(
        "La température doit être "
        "strictement positive."
    )


# ------------------------------------------------------------
# 6. RÉSUMÉ
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("REPRISE DU PIPELINE V9")
print("=" * 70)

print(
    "V9_DIR :",
    V9_DIR,
)

print(
    "FINAL_MODEL_DIR :",
    FINAL_MODEL_DIR,
)

print(
    "Poids LightGBM :",
    LGB_WEIGHT_FROM_METADATA,
)

print(
    "Poids Logistic :",
    LOGISTIC_WEIGHT_FROM_METADATA,
)

print(
    "Température :",
    TEMPERATURE_FROM_METADATA,
)

print(
    "\n✅ Tous les fichiers nécessaires "
    "au pipeline V9 sont disponibles."
)

Checkpoint V9 sélectionné :
/kaggle/input/notebooks/davoine/llm-arena-baseline-v1/v9_transformer_checkpoint

Dossier des modèles sélectionné :
/kaggle/input/datasets/davoine/donne-lui-un-nom-comme-llm-arena-final-inference/v9_final_models

REPRISE DU PIPELINE V9
V9_DIR : /kaggle/input/notebooks/davoine/llm-arena-baseline-v1/v9_transformer_checkpoint
FINAL_MODEL_DIR : /kaggle/input/datasets/davoine/donne-lui-un-nom-comme-llm-arena-final-inference/v9_final_models
Poids LightGBM : 0.94
Poids Logistic : 0.06
Température : 0.99

✅ Tous les fichiers nécessaires au pipeline V9 sont disponibles.


In [2]:
# ============================================================
# EXTRAIRE LE CODE SOURCE DE L’ANCIEN NOTEBOOK
# VERSION FINALE NÉCESSAIRE À L’INFÉRENCE
# ============================================================

from pathlib import Path
import json


INPUT_ROOT = Path("/kaggle/input")

OUTPUT_DIR = Path(
    "/kaggle/working/v9_pipeline_extraction"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 1. TROUVER LE FICHIER SOURCE DE L’ANCIEN NOTEBOOK
# ------------------------------------------------------------

source_candidates = []

patterns = [
    "**/__notebook_source__.ipynb",
    "**/*notebook_source*.ipynb",
    "**/*.ipynb",
    "**/__notebook_source__.py",
    "**/*notebook_source*.py",
]


for pattern in patterns:
    source_candidates.extend(
        INPUT_ROOT.glob(pattern)
    )


source_candidates = sorted(
    set(source_candidates),
    key=lambda path: (
        "llm-arena-baseline-v1"
        not in str(path).lower(),

        "__notebook_source__"
        not in path.name.lower(),

        len(str(path))
    )
)


if not source_candidates:
    raise FileNotFoundError(
        "Aucun fichier source de l’ancien notebook "
        "n’a été trouvé dans /kaggle/input."
    )


SOURCE_PATH = source_candidates[0]


print("✅ Source sélectionnée :")
print(SOURCE_PATH)


# ------------------------------------------------------------
# 2. LIRE ET EXTRAIRE LES CELLULES DE CODE
# ------------------------------------------------------------

try:
    raw_text = SOURCE_PATH.read_text(
        encoding="utf-8"
    )

except UnicodeDecodeError:
    raw_text = SOURCE_PATH.read_text(
        encoding="utf-8",
        errors="replace"
    )


code_cells = []


try:
    notebook_json = json.loads(
        raw_text
    )

    is_notebook = (
        isinstance(notebook_json, dict)
        and "cells" in notebook_json
    )

except Exception:
    notebook_json = None
    is_notebook = False


if is_notebook:

    for cell_index, cell in enumerate(
        notebook_json["cells"]
    ):
        if cell.get("cell_type") != "code":
            continue

        source = cell.get(
            "source",
            ""
        )

        if isinstance(source, list):
            source = "".join(source)
        else:
            source = str(source)

        if source.strip():
            code_cells.append({
                "cell_index": cell_index,
                "source": source
            })

else:

    code_cells = [{
        "cell_index": 0,
        "source": raw_text
    }]


if not code_cells:
    raise RuntimeError(
        "Aucune cellule de code n’a été extraite."
    )


print(
    "✅ Cellules de code extraites :",
    len(code_cells)
)


# ------------------------------------------------------------
# 3. CRÉER baseline_all_code.py
# ------------------------------------------------------------

all_code_parts = []


for cell in code_cells:

    all_code_parts.append(
        "\n\n"
        + "#" * 80
        + f"\n# CELLULE SOURCE {cell['cell_index']}\n"
        + "#" * 80
        + "\n\n"
        + cell["source"]
    )


all_code = "".join(
    all_code_parts
)

all_lines = all_code.splitlines()


ALL_CODE_PATH = (
    OUTPUT_DIR
    / "baseline_all_code.py"
)


ALL_CODE_PATH.write_text(
    all_code,
    encoding="utf-8"
)


if not ALL_CODE_PATH.exists():
    raise FileNotFoundError(
        "baseline_all_code.py n’a pas été créé."
    )


print("\n✅ Code complet exporté :")
print(ALL_CODE_PATH)

print(
    "Nombre total de lignes :",
    len(all_lines)
)

✅ Source sélectionnée :
/kaggle/input/notebooks/davoine/llm-arena-baseline-v1/.virtual_documents/__notebook_source__.ipynb
✅ Cellules de code extraites : 1

✅ Code complet exporté :
/kaggle/working/v9_pipeline_extraction/baseline_all_code.py
Nombre total de lignes : 34814


In [3]:
# ============================================================
# ÉTAPE 1 — PRÉPARER LE PIPELINE D’INFÉRENCE V9
# VERSION ROBUSTE ET SIMPLIFIÉE POUR LE TEST CACHÉ
# ============================================================

from pathlib import Path
import ast
import json

import pandas as pd


INPUT_ROOT = Path("/kaggle/input")


# ------------------------------------------------------------
# 1. VÉRIFIER LE DOSSIER DU CHECKPOINT V9
# ------------------------------------------------------------

if "V9_DIR" not in globals():
    raise NameError(
        "V9_DIR n'existe pas. La cellule qui retrouve "
        "le checkpoint V9 doit être exécutée avant cette étape."
    )


feature_lists_path = (
    Path(V9_DIR)
    / "feature_lists.json"
)


if not feature_lists_path.exists():
    raise FileNotFoundError(
        "feature_lists.json est introuvable : "
        f"{feature_lists_path}"
    )


# ------------------------------------------------------------
# 2. CHARGER LES LISTES EXACTES DE FEATURES
# ------------------------------------------------------------

with feature_lists_path.open(
    "r",
    encoding="utf-8"
) as file:
    V9_FEATURE_LISTS = json.load(file)


required_feature_list_keys = [
    "selected_features_v9_compact",
    "base_v8_features",
    "block_c_top_20",
    "block_b_top_20",
    "selected_block_a_features",
    "subq_features_5_of_5"
]


missing_feature_list_keys = [
    key
    for key in required_feature_list_keys
    if key not in V9_FEATURE_LISTS
]


if missing_feature_list_keys:
    raise KeyError(
        "Listes absentes de feature_lists.json : "
        + ", ".join(missing_feature_list_keys)
    )


SELECTED_FEATURES_V9 = list(
    V9_FEATURE_LISTS[
        "selected_features_v9_compact"
    ]
)

BASE_V8_FEATURES = list(
    V9_FEATURE_LISTS[
        "base_v8_features"
    ]
)

BLOCK_C_TOP_20 = list(
    V9_FEATURE_LISTS[
        "block_c_top_20"
    ]
)

BLOCK_B_TOP_20 = list(
    V9_FEATURE_LISTS[
        "block_b_top_20"
    ]
)

SELECTED_BLOCK_A_FEATURES = list(
    V9_FEATURE_LISTS[
        "selected_block_a_features"
    ]
)

SUBQ_FEATURES = list(
    V9_FEATURE_LISTS[
        "subq_features_5_of_5"
    ]
)


# Alias requis par les étapes suivantes
selected_features_v9_compact = (
    SELECTED_FEATURES_V9
)

base_v8_features = BASE_V8_FEATURES

block_c_top_20 = BLOCK_C_TOP_20

block_b_top_20 = BLOCK_B_TOP_20

selected_block_a_features = (
    SELECTED_BLOCK_A_FEATURES
)

subq_features_5_of_5 = (
    SUBQ_FEATURES
)


if len(SELECTED_FEATURES_V9) != 143:
    raise ValueError(
        "La liste V9 doit contenir exactement "
        f"143 features, pas {len(SELECTED_FEATURES_V9)}."
    )


if len(BASE_V8_FEATURES) != 123:
    raise ValueError(
        "base_v8_features doit contenir 123 features."
    )


if len(BLOCK_C_TOP_20) != 20:
    raise ValueError(
        "block_c_top_20 doit contenir 20 features."
    )


if len(BLOCK_B_TOP_20) != 20:
    raise ValueError(
        "block_b_top_20 doit contenir 20 features."
    )


if len(SELECTED_BLOCK_A_FEATURES) != 40:
    raise ValueError(
        "selected_block_a_features doit contenir "
        "40 features."
    )


if len(SUBQ_FEATURES) != 3:
    raise ValueError(
        "subq_features_5_of_5 doit contenir "
        "3 features."
    )


if (
    BASE_V8_FEATURES
    + BLOCK_C_TOP_20
    != SELECTED_FEATURES_V9
):
    raise ValueError(
        "L'ordre final V9 ne correspond pas à "
        "base_v8_features + block_c_top_20."
    )


print("✅ Listes exactes chargées")
print("V9 :", len(SELECTED_FEATURES_V9))
print("Bloc A :", len(SELECTED_BLOCK_A_FEATURES))
print("Bloc B :", len(BLOCK_B_TOP_20))
print("Bloc C :", len(BLOCK_C_TOP_20))
print("Sous-questions :", len(SUBQ_FEATURES))


# ------------------------------------------------------------
# 3. RETROUVER LE TEST DE LA COMPÉTITION
# ------------------------------------------------------------

expected_test_path = (
    INPUT_ROOT
    / "competitions"
    / "llm-classification-finetuning"
    / "test.csv"
)


if expected_test_path.exists():
    TEST_PATH = expected_test_path

else:
    test_candidates = [
        path
        for path in INPUT_ROOT.rglob("test.csv")
        if (
            "llm-classification-finetuning"
            in str(path).lower()
        )
    ]

    if not test_candidates:
        raise FileNotFoundError(
            "Le fichier test.csv de la compétition "
            "est introuvable."
        )

    TEST_PATH = sorted(
        test_candidates,
        key=lambda path: len(str(path))
    )[0]


test_raw = pd.read_csv(
    TEST_PATH
)


required_raw_columns = [
    "id",
    "prompt",
    "response_a",
    "response_b"
]


missing_raw_columns = [
    column
    for column in required_raw_columns
    if column not in test_raw.columns
]


if missing_raw_columns:
    raise KeyError(
        "Colonnes absentes du test : "
        + ", ".join(missing_raw_columns)
    )


if len(test_raw) == 0:
    raise ValueError(
        "Le fichier test.csv ne contient aucune ligne."
    )


# Ne conserver que les colonnes nécessaires.
test_raw = test_raw[
    required_raw_columns
].copy()


print("\n✅ Test brut chargé :")
print(TEST_PATH)
print("Dimensions :", test_raw.shape)


# ------------------------------------------------------------
# 4. PARSER LES CONVERSATIONS
# ------------------------------------------------------------

def parse_turns(value):
    """
    Convertit une conversation en liste de chaînes.
    Fonction tolérante aux valeurs absentes ou mal formatées.
    """

    if value is None:
        return []

    try:
        if pd.isna(value):
            return []
    except (
        TypeError,
        ValueError
    ):
        pass

    if isinstance(value, (list, tuple)):
        return [
            ""
            if item is None
            else str(item)
            for item in value
        ]

    text = str(value).strip()

    if not text:
        return []

    # Format JSON attendu par la compétition
    try:
        parsed = json.loads(text)

        if isinstance(parsed, list):
            return [
                ""
                if item is None
                else str(item)
                for item in parsed
            ]

        return [str(parsed)]

    except (
        json.JSONDecodeError,
        TypeError,
        ValueError
    ):
        pass

    # Solution de secours pour les listes Python
    try:
        parsed = ast.literal_eval(text)

        if isinstance(parsed, (list, tuple)):
            return [
                ""
                if item is None
                else str(item)
                for item in parsed
            ]

        return [str(parsed)]

    except (
        ValueError,
        SyntaxError,
        TypeError,
        MemoryError,
        RecursionError
    ):
        return [text]


def join_turns(turns):
    if not turns:
        return ""

    return "\n\n".join(
        str(turn)
        for turn in turns
    )


def get_last_turn(turns):
    if not turns:
        return ""

    return str(turns[-1])


test_inference = test_raw.copy()


for column in [
    "prompt",
    "response_a",
    "response_b"
]:
    turns_column = f"{column}_turns"
    full_column = f"{column}_full"
    last_column = f"{column}_last"
    count_column = f"{column}_turn_count"

    test_inference[turns_column] = (
        test_inference[column]
        .apply(parse_turns)
    )

    test_inference[full_column] = (
        test_inference[turns_column]
        .apply(join_turns)
    )

    test_inference[last_column] = (
        test_inference[turns_column]
        .apply(get_last_turn)
    )

    test_inference[count_column] = (
        test_inference[turns_column]
        .apply(len)
        .astype("int32")
    )


required_parsed_columns = [
    "prompt_full",
    "prompt_last",
    "response_a_full",
    "response_b_full",
    "prompt_turn_count",
    "response_a_turn_count",
    "response_b_turn_count"
]


missing_parsed_columns = [
    column
    for column in required_parsed_columns
    if column not in test_inference.columns
]


if missing_parsed_columns:
    raise RuntimeError(
        "Colonnes parsées absentes : "
        + ", ".join(missing_parsed_columns)
    )


if len(test_inference) != len(test_raw):
    raise ValueError(
        "Le parsing a changé le nombre de lignes."
    )


if not test_inference[
    "id"
].reset_index(
    drop=True
).equals(
    test_raw[
        "id"
    ].reset_index(
        drop=True
    )
):
    raise ValueError(
        "L'ordre des identifiants a changé "
        "pendant le parsing."
    )


print("\n✅ Conversations parsées")

display(
    test_inference[
        [
            "id",
            "prompt_turn_count",
            "response_a_turn_count",
            "response_b_turn_count"
        ]
    ].head()
)


# ------------------------------------------------------------
# 5. TROUVER UNIQUEMENT LE MODÈLE OFFICIEL LOCAL
# ------------------------------------------------------------

embedding_candidates = []


for modules_path in INPUT_ROOT.rglob(
    "modules.json"
):
    candidate = modules_path.parent

    candidate_text = str(
        candidate
    ).lower()

    if (
        "official-minilm-l12-v2-v9"
        in candidate_text
        or
        "official_paraphrase_multilingual"
        in candidate_text
    ):
        embedding_candidates.append(
            candidate
        )


embedding_candidates = sorted(
    set(embedding_candidates),
    key=lambda path: len(str(path))
)


if not embedding_candidates:
    raise FileNotFoundError(
        "Le modèle officiel validé n'a pas été "
        "trouvé dans les Inputs Kaggle."
    )


EMBEDDING_MODEL_PATH = (
    embedding_candidates[0]
)


required_model_paths = [
    EMBEDDING_MODEL_PATH
    / "modules.json"
]


missing_model_paths = [
    path
    for path in required_model_paths
    if not path.exists()
]


if missing_model_paths:
    raise FileNotFoundError(
        "Le dossier du modèle officiel est incomplet : "
        + ", ".join(
            str(path)
            for path in missing_model_paths
        )
    )


print("\n✅ Modèle officiel local trouvé :")
print(EMBEDDING_MODEL_PATH)


# ------------------------------------------------------------
# 6. RÉSUMÉ
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RÉSUMÉ DE PRÉPARATION")
print("=" * 70)

print("Test brut :", test_raw.shape)
print("Test parsé :", test_inference.shape)

print(
    "Features finales attendues :",
    len(selected_features_v9_compact)
)

print(
    "Modèle d'embeddings :",
    EMBEDDING_MODEL_PATH
)

print(
    "\n✅ Étape 1 terminée et prête "
    "pour le test public ou caché."
)

✅ Listes exactes chargées
V9 : 143
Bloc A : 40
Bloc B : 20
Bloc C : 20
Sous-questions : 3

✅ Test brut chargé :
/kaggle/input/competitions/llm-classification-finetuning/test.csv
Dimensions : (3, 4)

✅ Conversations parsées


,id,prompt_turn_count,response_a_turn_count,response_b_turn_count
0,136060,1,1,1
1,211333,1,1,1
2,1233961,2,2,2



✅ Modèle officiel local trouvé :
/kaggle/input/datasets/davoine/official-minilm-l12-v2-v9

RÉSUMÉ DE PRÉPARATION
Test brut : (3, 4)
Test parsé : (3, 16)
Features finales attendues : 143
Modèle d'embeddings : /kaggle/input/datasets/davoine/official-minilm-l12-v2-v9

✅ Étape 1 terminée et prête pour le test public ou caché.


In [4]:
# ============================================================
# ÉTAPE 2 — CALCULER LES 9 FEATURES SÉMANTIQUES DU TEST
# VERSION PAR LOTS, PLUS ROBUSTE POUR LE TEST CACHÉ
# ============================================================

import gc
import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer


# ------------------------------------------------------------
# 1. CHARGER LE MODÈLE LOCAL
# ------------------------------------------------------------

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Appareil utilisé :", DEVICE)
print("Modèle local :", EMBEDDING_MODEL_PATH)


embedding_model = SentenceTransformer(
    str(EMBEDDING_MODEL_PATH),
    device=DEVICE
)


if hasattr(
    embedding_model,
    "get_embedding_dimension"
):
    embedding_dimension = (
        embedding_model.get_embedding_dimension()
    )
else:
    embedding_dimension = (
        embedding_model
        .get_sentence_embedding_dimension()
    )


print(
    "Dimension des embeddings :",
    embedding_dimension
)


# ------------------------------------------------------------
# 2. PARAMÈTRES MÉMOIRE
# ------------------------------------------------------------

MAX_TEXT_CHARACTERS = 6000

# Nombre de duels traités simultanément.
# Valeur prudente pour le test caché.
ROW_CHUNK_SIZE = 64

# Taille interne d'encodage.
ENCODE_BATCH_SIZE = 16


# ------------------------------------------------------------
# 3. PRÉPARER LES TEXTES
# ------------------------------------------------------------

def prepare_embedding_text(value):
    if value is None:
        return ""

    if (
        isinstance(value, float)
        and pd.isna(value)
    ):
        return ""

    text = str(value).strip()

    if len(text) <= MAX_TEXT_CHARACTERS:
        return text

    half = MAX_TEXT_CHARACTERS // 2

    return (
        text[:half]
        + "\n[...]\n"
        + text[-half:]
    )


def encode_text_list(texts):
    prepared_texts = [
        prepare_embedding_text(value)
        for value in texts
    ]

    embeddings = embedding_model.encode(
        prepared_texts,
        batch_size=ENCODE_BATCH_SIZE,
        show_progress_bar=False,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    return embeddings.astype(
        "float32"
    )


def rowwise_cosine(
    embeddings_1,
    embeddings_2
):
    return np.sum(
        embeddings_1 * embeddings_2,
        axis=1
    )


# ------------------------------------------------------------
# 4. CALCULER LES FEATURES PAR LOTS
# ------------------------------------------------------------

semantic_chunks = []

number_of_rows = len(
    test_inference
)

print(
    "Nombre de lignes à encoder :",
    number_of_rows
)


for start_index in range(
    0,
    number_of_rows,
    ROW_CHUNK_SIZE
):
    end_index = min(
        start_index + ROW_CHUNK_SIZE,
        number_of_rows
    )

    chunk = test_inference.iloc[
        start_index:end_index
    ]


    prompt_full_embeddings = encode_text_list(
        chunk["prompt_full"].tolist()
    )

    prompt_last_embeddings = encode_text_list(
        chunk["prompt_last"].tolist()
    )

    response_a_embeddings = encode_text_list(
        chunk["response_a_full"].tolist()
    )

    response_b_embeddings = encode_text_list(
        chunk["response_b_full"].tolist()
    )


    semantic_full_a = rowwise_cosine(
        prompt_full_embeddings,
        response_a_embeddings
    )

    semantic_full_b = rowwise_cosine(
        prompt_full_embeddings,
        response_b_embeddings
    )

    semantic_last_a = rowwise_cosine(
        prompt_last_embeddings,
        response_a_embeddings
    )

    semantic_last_b = rowwise_cosine(
        prompt_last_embeddings,
        response_b_embeddings
    )

    semantic_response_ab = rowwise_cosine(
        response_a_embeddings,
        response_b_embeddings
    )


    semantic_chunk = pd.DataFrame({
        "semantic_prompt_full_a":
            semantic_full_a,

        "semantic_prompt_full_b":
            semantic_full_b,

        "semantic_delta_prompt_full":
            semantic_full_a
            - semantic_full_b,

        "semantic_abs_delta_prompt_full":
            np.abs(
                semantic_full_a
                - semantic_full_b
            ),

        "semantic_prompt_last_a":
            semantic_last_a,

        "semantic_prompt_last_b":
            semantic_last_b,

        "semantic_delta_prompt_last":
            semantic_last_a
            - semantic_last_b,

        "semantic_abs_delta_prompt_last":
            np.abs(
                semantic_last_a
                - semantic_last_b
            ),

        "semantic_response_ab":
            semantic_response_ab
    }).astype("float32")


    semantic_chunks.append(
        semantic_chunk
    )


    # Libérer immédiatement la mémoire du lot
    del prompt_full_embeddings
    del prompt_last_embeddings
    del response_a_embeddings
    del response_b_embeddings

    del semantic_full_a
    del semantic_full_b
    del semantic_last_a
    del semantic_last_b
    del semantic_response_ab

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    print(
        f"✅ Embeddings traités : "
        f"{end_index}/{number_of_rows}"
    )


# ------------------------------------------------------------
# 5. ASSEMBLER LES LOTS
# ------------------------------------------------------------

if not semantic_chunks:
    raise RuntimeError(
        "Aucun lot sémantique n'a été créé."
    )


test_semantic_features = pd.concat(
    semantic_chunks,
    ignore_index=True
)


del semantic_chunks

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ------------------------------------------------------------
# 6. VÉRIFICATIONS
# ------------------------------------------------------------

EXPECTED_SEMANTIC_COLUMNS = [
    "semantic_prompt_full_a",
    "semantic_prompt_full_b",
    "semantic_delta_prompt_full",
    "semantic_abs_delta_prompt_full",
    "semantic_prompt_last_a",
    "semantic_prompt_last_b",
    "semantic_delta_prompt_last",
    "semantic_abs_delta_prompt_last",
    "semantic_response_ab"
]


if list(
    test_semantic_features.columns
) != EXPECTED_SEMANTIC_COLUMNS:
    raise ValueError(
        "Les colonnes sémantiques "
        "ne sont pas dans le bon ordre."
    )


if test_semantic_features.shape != (
    number_of_rows,
    9
):
    raise ValueError(
        "Dimensions incorrectes pour "
        "les features sémantiques : "
        f"{test_semantic_features.shape}"
    )


if (
    test_semantic_features
    .isna()
    .sum()
    .sum()
    != 0
):
    raise ValueError(
        "Les features sémantiques "
        "contiennent des valeurs manquantes."
    )


if not np.isfinite(
    test_semantic_features.to_numpy()
).all():
    raise ValueError(
        "Les features sémantiques "
        "contiennent des valeurs infinies."
    )


print(
    "\n✅ Features sémantiques créées :",
    test_semantic_features.shape
)

print(
    "✅ Les 9 colonnes sont valides "
    "pour le test public ou caché."
)

display(
    test_semantic_features.head()
)


# ------------------------------------------------------------
# 7. LIBÉRER LE MODÈLE SI PLUS NÉCESSAIRE
# ------------------------------------------------------------

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


print(
    "✅ Calcul sémantique terminé "
    "avec gestion mémoire par lots."
)

Appareil utilisé : cpu
Modèle local : /kaggle/input/datasets/davoine/official-minilm-l12-v2-v9


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Dimension des embeddings : 384
Nombre de lignes à encoder : 3
✅ Embeddings traités : 3/3

✅ Features sémantiques créées : (3, 9)
✅ Les 9 colonnes sont valides pour le test public ou caché.


,semantic_prompt_full_a,semantic_prompt_full_b,semantic_delta_prompt_full,semantic_abs_delta_prompt_full,semantic_prompt_last_a,semantic_prompt_last_b,semantic_delta_prompt_last,semantic_abs_delta_prompt_last,semantic_response_ab
0,0.692824,0.752063,-0.059240,0.059240,0.692824,0.752063,-0.059240,0.059240,0.718736
1,0.441458,0.788541,-0.347083,0.347083,0.441458,0.788541,-0.347083,0.347083,0.425148
2,0.837017,0.895574,-0.058558,0.058558,0.249467,0.260560,-0.011093,0.011093,0.876496


✅ Calcul sémantique terminé avec gestion mémoire par lots.


In [5]:
# ============================================================
# COMPARER LES TEXTES ACTUELS AUX TEXTES DU CHECKPOINT
# ============================================================

saved_test_texts = pd.read_parquet(
    V9_DIR / "test_texts.parquet"
)

columns_to_compare = [
    "prompt_full",
    "prompt_last",
    "response_a_full",
    "response_b_full"
]

print("Dimensions textes sauvegardés :", saved_test_texts.shape)
print("Dimensions textes actuels :", test_inference.shape)

for column in columns_to_compare:
    current_values = (
        test_inference[column]
        .astype(str)
        .reset_index(drop=True)
    )

    saved_values = (
        saved_test_texts[column]
        .astype(str)
        .reset_index(drop=True)
    )

    identical = current_values.equals(saved_values)

    print(f"{column} identique :", identical)

    if not identical:
        for row_index in range(
            min(len(current_values), len(saved_values))
        ):
            if current_values.iloc[row_index] != saved_values.iloc[row_index]:
                print(
                    f"\nPremière différence, ligne {row_index}"
                )
                print(
                    "ACTUEL :",
                    repr(current_values.iloc[row_index][:500])
                )
                print(
                    "SAUVEGARDÉ :",
                    repr(saved_values.iloc[row_index][:500])
                )
                break

Dimensions textes sauvegardés : (3, 4)
Dimensions textes actuels : (3, 16)
prompt_full identique : True
prompt_last identique : True
response_a_full identique : True
response_b_full identique : True


In [6]:
# ============================================================
# RECHERCHER LE MODÈLE D'EMBEDDINGS EXACT DANS LES INPUTS
# ============================================================

from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

keywords = [
    "paraphrase-multilingual",
    "minilm-l12",
    "sentence_transformer",
    "sentence-transformers",
    "embedding_model"
]

matches = []

for path in INPUT_ROOT.rglob("*"):
    path_text = str(path).lower()

    if any(keyword in path_text for keyword in keywords):
        matches.append(path)

print("Éléments trouvés :", len(matches))

for path in matches[:300]:
    print(path)

Éléments trouvés : 47
/kaggle/input/datasets/davoine/official-minilm-l12-v2-v9
/kaggle/input/datasets/davoine/official-minilm-l12-v2-v9/config.json
/kaggle/input/datasets/davoine/official-minilm-l12-v2-v9/1_Pooling
/kaggle/input/datasets/davoine/official-minilm-l12-v2-v9/README.md
/kaggle/input/datasets/davoine/official-minilm-l12-v2-v9/tokenizer.json
/kaggle/input/datasets/davoine/official-minilm-l12-v2-v9/tokenizer_config.json
/kaggle/input/datasets/davoine/official-minilm-l12-v2-v9/sentence_bert_config.json
/kaggle/input/datasets/davoine/official-minilm-l12-v2-v9/config_sentence_transformers.json
/kaggle/input/datasets/davoine/official-minilm-l12-v2-v9/model.safetensors
/kaggle/input/datasets/davoine/official-minilm-l12-v2-v9/modules.json
/kaggle/input/datasets/davoine/official-minilm-l12-v2-v9/1_Pooling/config.json
/kaggle/input/models/puffinlynx/paraphrase-multilingual-minilm-l12-v2
/kaggle/input/models/puffinlynx/paraphrase-multilingual-minilm-l12-v2/pytorch
/kaggle/input/models/

In [7]:
# ============================================================
# RECHERCHER DES DOSSIERS DE MODÈLE COMPLETS
# ============================================================

model_dirs = []

for config_path in INPUT_ROOT.rglob("modules.json"):
    model_dirs.append(config_path.parent)

for config_path in INPUT_ROOT.rglob("sentence_bert_config.json"):
    model_dirs.append(config_path.parent)

for config_path in INPUT_ROOT.rglob("config_sentence_transformers.json"):
    model_dirs.append(config_path.parent)

model_dirs = sorted(set(model_dirs))

print("Dossiers SentenceTransformer trouvés :")

for path in model_dirs:
    print("-", path)

Dossiers SentenceTransformer trouvés :
- /kaggle/input/datasets/davoine/official-minilm-l12-v2-v9
- /kaggle/input/models/puffinlynx/paraphrase-multilingual-minilm-l12-v2/pytorch/default/1/paraphrase-multilingual-MiniLM-L12-v2


In [8]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

candidates = []

for path in INPUT_ROOT.rglob("modules.json"):
    model_dir = path.parent

    if (
        "official-minilm" in str(model_dir).lower()
        or "official_paraphrase" in str(model_dir).lower()
    ):
        candidates.append(model_dir)

print("Modèles officiels trouvés :")

for path in candidates:
    print("✅", path)

if not candidates:
    raise FileNotFoundError(
        "Le modèle officiel n'a pas été trouvé dans /kaggle/input."
    )

OFFICIAL_INPUT_MODEL_PATH = candidates[0]

print("\nChemin permanent utilisé :")
print(OFFICIAL_INPUT_MODEL_PATH)

Modèles officiels trouvés :
✅ /kaggle/input/datasets/davoine/official-minilm-l12-v2-v9

Chemin permanent utilisé :
/kaggle/input/datasets/davoine/official-minilm-l12-v2-v9


In [9]:
# ============================================================
# ÉTAPE 3 — CHARGER LES FONCTIONS ET CONSTANTES DE FEATURES
# VERSION ROBUSTE, HORS LIGNE ET COMPATIBLE TEST CACHÉ
# ============================================================

from pathlib import Path
from collections import Counter, defaultdict

import ast
import json
import math
import os
import re

import numpy as np
import pandas as pd

from tqdm.auto import tqdm


# ------------------------------------------------------------
# 1. RETROUVER baseline_all_code.py
# ------------------------------------------------------------

def find_baseline_source():
    preferred_candidates = []

    for root in [
        Path("/kaggle/working"),
        Path("/kaggle/input"),
    ]:
        if not root.exists():
            continue

        for path in root.rglob(
            "baseline_all_code.py"
        ):
            if not path.is_file():
                continue

            preferred_candidates.append(path)

    preferred_candidates = sorted(
        set(preferred_candidates),
        key=lambda path: (
            "v9_pipeline_extraction"
            not in str(path).lower(),
            "/kaggle/working/"
            not in str(path).lower(),
            len(str(path)),
            str(path),
        ),
    )

    return (
        preferred_candidates[0]
        if preferred_candidates
        else None
    )


BASELINE_SOURCE_PATH = find_baseline_source()


if BASELINE_SOURCE_PATH is None:
    raise FileNotFoundError(
        "baseline_all_code.py est introuvable."
    )


print("✅ Code source trouvé :")
print(BASELINE_SOURCE_PATH)


# ------------------------------------------------------------
# 2. NETTOYER L’EXPORT DE NOTEBOOK
# ------------------------------------------------------------

raw_source = BASELINE_SOURCE_PATH.read_text(
    encoding="utf-8",
    errors="replace",
)


clean_lines = []


for line in raw_source.splitlines():
    stripped = line.strip()

    # Commandes de notebook ou shell
    if (
        "get_ipython()" in line
        or stripped.startswith("%")
        or stripped.startswith("!")
    ):
        continue

    # Outputs accidentellement intégrés au script
    if stripped.startswith(
        (
            "Loading weights:",
            "Batches:",
            "Requirement already satisfied:",
            "Collecting ",
            "Downloading ",
            "Installing collected packages:",
            "Successfully installed ",
            "Retrying in ",
            "[Errno ",
        )
    ):
        continue

    clean_lines.append(line)


clean_source = "\n".join(clean_lines)


# ------------------------------------------------------------
# 3. RENDRE LE SCRIPT ANALYSABLE PAR AST
# ------------------------------------------------------------

def parse_with_progressive_cleanup(
    source,
    maximum_attempts=40,
):
    current_source = source

    for attempt in range(
        maximum_attempts
    ):
        try:
            tree = ast.parse(
                current_source
            )

            return tree, current_source

        except SyntaxError as error:
            if error.lineno is None:
                raise

            lines = current_source.splitlines()

            bad_line_index = max(
                error.lineno - 1,
                0,
            )

            start = max(
                bad_line_index - 3,
                0,
            )

            end = min(
                bad_line_index + 4,
                len(lines),
            )

            print(
                f"Nettoyage {attempt + 1} : "
                f"lignes {start + 1} à {end} neutralisées"
            )

            for index in range(
                start,
                end,
            ):
                lines[index] = (
                    "# LIGNE EXPORT NOTEBOOK IGNORÉE"
                )

            current_source = "\n".join(
                lines
            )

    raise RuntimeError(
        "Impossible de rendre le fichier source "
        "analysable automatiquement."
    )


tree, cleaned_source = (
    parse_with_progressive_cleanup(
        clean_source
    )
)


print(
    "\n✅ Le code source nettoyé "
    "est maintenant analysable"
)


# ------------------------------------------------------------
# 4. CRÉER LE NAMESPACE CONTRÔLÉ
# ------------------------------------------------------------

feature_namespace = {
    "__name__":
        "feature_extraction_only",

    # Modules et objets utilisés par les fonctions
    "ast": ast,
    "json": json,
    "math": math,
    "os": os,
    "re": re,

    "np": np,
    "pd": pd,

    "Counter": Counter,
    "defaultdict": defaultdict,
    "tqdm": tqdm,
}


# ------------------------------------------------------------
# 5. CHARGER LES IMPORTS SÛRS DU VIEUX SCRIPT
# ------------------------------------------------------------

blocked_import_roots = {
    "sentence_transformers",
    "transformers",
    "huggingface_hub",
    "tensorflow",
    "keras",
}


for node in tree.body:
    if not isinstance(
        node,
        (
            ast.Import,
            ast.ImportFrom,
        ),
    ):
        continue

    if isinstance(node, ast.Import):
        roots = {
            alias.name.split(".")[0]
            for alias in node.names
        }

    else:
        roots = {
            (
                node.module.split(".")[0]
                if node.module
                else ""
            )
        }

    if roots & blocked_import_roots:
        continue

    try:
        module = ast.Module(
            body=[node],
            type_ignores=[],
        )

        exec(
            compile(
                module,
                filename=str(
                    BASELINE_SOURCE_PATH
                ),
                mode="exec",
            ),
            feature_namespace,
        )

    except Exception:
        # Import facultatif inutile à l’inférence
        pass


# ------------------------------------------------------------
# 6. CHARGER TOUTES LES FONCTIONS
# ------------------------------------------------------------

loaded_functions = []


for node in tree.body:
    if not isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        ),
    ):
        continue

    try:
        module = ast.Module(
            body=[node],
            type_ignores=[],
        )

        exec(
            compile(
                module,
                filename=str(
                    BASELINE_SOURCE_PATH
                ),
                mode="exec",
            ),
            feature_namespace,
        )

        loaded_functions.append(
            node.name
        )

    except Exception as error:
        print(
            "Fonction non chargée :",
            node.name,
            "|",
            type(error).__name__,
        )


print(
    "\n✅ Fonctions chargées :",
    len(loaded_functions),
)


# ------------------------------------------------------------
# 7. OUTILS POUR FILTRER LES CONSTANTES
# ------------------------------------------------------------

def get_assigned_names(node):
    if isinstance(node, ast.Assign):
        targets = node.targets

    elif isinstance(
        node,
        ast.AnnAssign,
    ):
        targets = [node.target]

    else:
        return []

    names = []

    for target in targets:
        if isinstance(target, ast.Name):
            names.append(target.id)

        elif isinstance(
            target,
            (
                ast.Tuple,
                ast.List,
            ),
        ):
            for element in target.elts:
                if isinstance(
                    element,
                    ast.Name,
                ):
                    names.append(
                        element.id
                    )

    return names


def get_call_name(call_node):
    function = call_node.func

    if isinstance(function, ast.Name):
        return function.id

    if isinstance(
        function,
        ast.Attribute,
    ):
        parts = []
        current = function

        while isinstance(
            current,
            ast.Attribute,
        ):
            parts.append(current.attr)
            current = current.value

        if isinstance(current, ast.Name):
            parts.append(current.id)

        return ".".join(
            reversed(parts)
        )

    return ""


forbidden_call_fragments = {
    "SentenceTransformer",
    "AutoTokenizer",
    "AutoModel",
    "from_pretrained",
    "snapshot_download",
    "hf_hub_download",
    "read_csv",
    "read_parquet",
    "read_json",
    "joblib.load",
    "torch.load",
    "fit",
    "fit_transform",
    "predict",
    "predict_proba",
    "encode",
    "to_csv",
    "to_parquet",
    "make_archive",
    "write_text",
}


def assignment_is_safe(node):
    for child in ast.walk(node):
        if not isinstance(
            child,
            ast.Call,
        ):
            continue

        call_name = get_call_name(
            child
        ).lower()

        if any(
            fragment.lower()
            in call_name
            for fragment
            in forbidden_call_fragments
        ):
            return False

    return True


# ------------------------------------------------------------
# 8. CHARGER LES CONSTANTES ET PATTERNS SÛRS
# ------------------------------------------------------------

safe_assignment_nodes = []


for node in tree.body:
    if not isinstance(
        node,
        (
            ast.Assign,
            ast.AnnAssign,
        ),
    ):
        continue

    names = get_assigned_names(node)

    looks_useful = any(
        name.isupper()
        or name.endswith("_PATTERN")
        or name.endswith("_PATTERNS")
        or name.endswith("_FEATURE_NAMES")
        for name in names
    )

    if (
        looks_useful
        and assignment_is_safe(node)
    ):
        safe_assignment_nodes.append(
            node
        )


remaining_nodes = list(
    safe_assignment_nodes
)


for pass_number in range(30):
    next_remaining = []
    loaded_this_pass = 0

    for node in remaining_nodes:
        try:
            module = ast.Module(
                body=[node],
                type_ignores=[],
            )

            exec(
                compile(
                    module,
                    filename=str(
                        BASELINE_SOURCE_PATH
                    ),
                    mode="exec",
                ),
                feature_namespace,
            )

            loaded_this_pass += 1

        except Exception:
            next_remaining.append(node)

    remaining_nodes = next_remaining

    print(
        f"Passage {pass_number + 1} : "
        f"{loaded_this_pass} constantes chargées, "
        f"{len(remaining_nodes)} restantes"
    )

    if loaded_this_pass == 0:
        break


# ------------------------------------------------------------
# 9. RECONSTRUIRE LES CONSTANTES PRINCIPALES
# ------------------------------------------------------------

if "text_statistics" not in feature_namespace:
    raise NameError(
        "text_statistics est absente."
    )


feature_namespace[
    "RESPONSE_FEATURE_NAMES"
] = list(
    feature_namespace[
        "text_statistics"
    ]("").keys()
)


feature_namespace[
    "PROMPT_FEATURE_NAMES"
] = [
    "word_count",
    "char_count",
    "sentence_count",
    "question_count",
    "bullet_count",
    "code_block_count",
]


if (
    "advanced_text_statistics"
    not in feature_namespace
):
    raise NameError(
        "advanced_text_statistics est absente."
    )


feature_namespace[
    "ADVANCED_FEATURE_NAMES"
] = list(
    feature_namespace[
        "advanced_text_statistics"
    ]("").keys()
)


if (
    "readability_statistics"
    not in feature_namespace
):
    raise NameError(
        "readability_statistics est absente."
    )


feature_namespace[
    "READABILITY_FEATURE_NAMES"
] = list(
    feature_namespace[
        "readability_statistics"
    ]("").keys()
)


# ------------------------------------------------------------
# 10. CORRIGER code_technical_statistics
# ------------------------------------------------------------

technical_function_node = next(
    (
        node
        for node in tree.body
        if (
            isinstance(
                node,
                ast.FunctionDef,
            )
            and node.name
            == "code_technical_statistics"
        )
    ),
    None,
)


if technical_function_node is None:
    raise RuntimeError(
        "code_technical_statistics "
        "est introuvable."
    )


cleaned_lines = (
    cleaned_source.splitlines()
)


technical_source = "\n".join(
    cleaned_lines[
        technical_function_node.lineno - 1:
        technical_function_node.end_lineno
    ]
)


assignment_start = (
    technical_source.find(
        "    assignment_symbol_count = len("
    )
)

comparison_start = (
    technical_source.find(
        "    comparison_symbol_count = len("
    )
)


if (
    assignment_start != -1
    and comparison_start != -1
    and comparison_start > assignment_start
):
    corrected_assignment_block = '''
    assignment_symbol_count = len(
        re.findall(
            r"(?<![=!<>])=(?!=)",
            code_like_text
        )
    )

'''

    technical_source = (
        technical_source[
            :assignment_start
        ]
        + corrected_assignment_block
        + technical_source[
            comparison_start:
        ]
    )


exec(
    compile(
        technical_source,
        filename=(
            "<code_technical_statistics_fixed>"
        ),
        mode="exec",
    ),
    feature_namespace,
)


# ------------------------------------------------------------
# 11. CRÉER TECH_FEATURE_NAMES
# ------------------------------------------------------------

feature_namespace[
    "TECH_FEATURE_NAMES"
] = list(
    feature_namespace[
        "code_technical_statistics"
    ](
        response="",
        prompt="",
    ).keys()
)


# ------------------------------------------------------------
# 12. VÉRIFIER LES OBJETS NÉCESSAIRES
# ------------------------------------------------------------

required_feature_functions = [
    "build_feature_table",
    "add_shorter_binary_features",
    "build_compact_pairwise_features",
    "build_constraint_feature_table",
    "build_advanced_feature_table",
    "build_readability_feature_table",
    "build_subquestion_feature_table",
    "build_reasoning_factuality_table",
    "build_code_technical_table",
    "build_tone_safety_role_table",
    "swap_v9_features",
]


missing_feature_functions = [
    name
    for name
    in required_feature_functions
    if name not in feature_namespace
]


if missing_feature_functions:
    raise RuntimeError(
        "Fonctions nécessaires absentes : "
        + ", ".join(
            missing_feature_functions
        )
    )


required_constants = [
    "RESPONSE_FEATURE_NAMES",
    "PROMPT_FEATURE_NAMES",
    "ADVANCED_FEATURE_NAMES",
    "READABILITY_FEATURE_NAMES",
    "TECH_FEATURE_NAMES",
]


missing_constants = [
    name
    for name in required_constants
    if name not in feature_namespace
]


if missing_constants:
    raise RuntimeError(
        "Constantes nécessaires absentes : "
        + ", ".join(
            missing_constants
        )
    )


print(
    "\n✅ Toutes les fonctions et constantes "
    "nécessaires au pipeline V9 sont disponibles"
)


for name in required_constants:
    print(
        "✅",
        name,
        "→",
        len(feature_namespace[name]),
        "éléments",
    )


print(
    "✅ Aucun modèle en ligne n’a été chargé "
    "depuis le vieux script"
)

✅ Code source trouvé :
/kaggle/working/v9_pipeline_extraction/baseline_all_code.py

✅ Le code source nettoyé est maintenant analysable

✅ Fonctions chargées : 220
Passage 1 : 185 constantes chargées, 33 restantes
Passage 2 : 0 constantes chargées, 33 restantes

✅ Toutes les fonctions et constantes nécessaires au pipeline V9 sont disponibles
✅ RESPONSE_FEATURE_NAMES → 20 éléments
✅ PROMPT_FEATURE_NAMES → 6 éléments
✅ ADVANCED_FEATURE_NAMES → 17 éléments
✅ READABILITY_FEATURE_NAMES → 20 éléments
✅ TECH_FEATURE_NAMES → 47 éléments
✅ Aucun modèle en ligne n’a été chargé depuis le vieux script


In [10]:
# ============================================================
# PROTECTION DES CONSTRUCTEURS CONTRE LES LIGNES PROBLÉMATIQUES
# VERSION ROBUSTE POUR LE TEST CACHÉ
# ============================================================

import numpy as np
import pandas as pd


def build_table_safely(
    builder_function,
    dataframe,
    description,
    expected_columns=None,
):
    """
    Essaie d'abord le traitement global.

    En cas d'erreur :
    - traite les lignes une par une ;
    - conserve les lignes valides ;
    - remplace une ligne problématique par des zéros.
    """

    try:
        result = builder_function(
            dataframe,
            description,
        )

        if not isinstance(
            result,
            pd.DataFrame,
        ):
            raise TypeError(
                f"{description} n'a pas renvoyé "
                "un DataFrame."
            )

        if len(result) != len(dataframe):
            raise ValueError(
                f"{description} a produit "
                f"{len(result)} lignes au lieu de "
                f"{len(dataframe)}."
            )

        return result.reset_index(
            drop=True
        )

    except Exception as full_error:
        print(
            f"⚠️ {description} a échoué globalement : "
            f"{type(full_error).__name__}. "
            "Passage ligne par ligne."
        )


    row_results = []

    discovered_columns = (
        list(expected_columns)
        if expected_columns is not None
        else None
    )

    failed_row_count = 0


    for row_index in range(
        len(dataframe)
    ):
        one_row = dataframe.iloc[
            [row_index]
        ].copy()

        try:
            row_result = builder_function(
                one_row,
                f"{description} ligne {row_index}",
            )

            if not isinstance(
                row_result,
                pd.DataFrame,
            ):
                raise TypeError(
                    "Le constructeur n'a pas "
                    "renvoyé un DataFrame."
                )

            row_result = row_result.reset_index(
                drop=True
            )

            if len(row_result) != 1:
                raise ValueError(
                    "Le constructeur n'a pas "
                    "renvoyé exactement une ligne."
                )

            if discovered_columns is None:
                discovered_columns = (
                    row_result.columns.tolist()
                )

            row_result = row_result.reindex(
                columns=discovered_columns,
                fill_value=0.0,
            )

            row_results.append(
                row_result
            )

        except Exception:
            failed_row_count += 1

            row_results.append(
                None
            )


    if discovered_columns is None:
        raise RuntimeError(
            f"Aucune ligne valide n'a permis "
            f"de déterminer les colonnes pour "
            f"{description}."
        )


    zero_row = pd.DataFrame(
        np.zeros(
            (
                1,
                len(discovered_columns),
            ),
            dtype=np.float32,
        ),
        columns=discovered_columns,
    )


    row_results = [
        (
            result
            if result is not None
            else zero_row.copy()
        )
        for result in row_results
    ]


    final_result = pd.concat(
        row_results,
        ignore_index=True,
    )


    final_result = final_result.reindex(
        columns=discovered_columns,
        fill_value=0.0,
    )


    if len(final_result) != len(dataframe):
        raise RuntimeError(
            f"{description} a produit "
            "un nombre de lignes incorrect."
        )


    final_result = final_result.replace(
        [np.inf, -np.inf],
        np.nan,
    ).fillna(0.0)


    if failed_row_count:
        print(
            f"⚠️ {description} : "
            f"{failed_row_count} ligne(s) "
            "remplacée(s) par des zéros."
        )
    else:
        print(
            f"✅ {description} terminé "
            "en mode ligne par ligne."
        )


    return final_result

In [11]:
# ============================================================
# CONSTRUCTION PAIRWISE DÉTERMINISTE
# Ne dépend pas des valeurs du test caché
# ============================================================

import numpy as np
import pandas as pd


def build_compact_pairwise_deterministic(
    feature_df
):
    if not isinstance(
        feature_df,
        pd.DataFrame
    ):
        raise TypeError(
            "feature_df doit être un DataFrame."
        )

    compact = feature_df.copy()
    columns_to_drop = []

    original_columns = list(
        feature_df.columns
    )

    # --------------------------------------------------------
    # 1. Couples a_feature / b_feature
    # --------------------------------------------------------

    for a_column in original_columns:
        if not a_column.startswith("a_"):
            continue

        suffix = a_column[2:]
        b_column = f"b_{suffix}"

        if b_column not in feature_df.columns:
            continue

        if suffix == "is_shorter_words":
            continue

        mean_column = f"mean_{suffix}"
        delta_column = f"delta_{suffix}"

        compact[mean_column] = (
            feature_df[a_column]
            + feature_df[b_column]
        ) / 2.0

        if delta_column not in compact.columns:
            compact[delta_column] = (
                feature_df[a_column]
                - feature_df[b_column]
            )

        columns_to_drop.extend([
            a_column,
            b_column
        ])

    # --------------------------------------------------------
    # 2. Couples feature_a / feature_b
    # --------------------------------------------------------

    for a_column in original_columns:
        if not a_column.endswith("_a"):
            continue

        base_name = a_column[:-2]
        b_column = f"{base_name}_b"

        if b_column not in feature_df.columns:
            continue

        mean_column = f"mean_{base_name}"
        delta_column = f"delta_{base_name}"

        compact[mean_column] = (
            feature_df[a_column]
            + feature_df[b_column]
        ) / 2.0

        if delta_column not in compact.columns:
            compact[delta_column] = (
                feature_df[a_column]
                - feature_df[b_column]
            )

        columns_to_drop.extend([
            a_column,
            b_column
        ])

    # --------------------------------------------------------
    # 3. Variables binaires
    # --------------------------------------------------------

    if "b_is_shorter_words" in compact.columns:
        columns_to_drop.append(
            "b_is_shorter_words"
        )

    # --------------------------------------------------------
    # 4. Nombre de tours
    # --------------------------------------------------------

    if "delta_turn_count" in compact.columns:
        compact[
            "abs_delta_turn_count"
        ] = compact[
            "delta_turn_count"
        ].abs()

        columns_to_drop.extend([
            "delta_response_turn_count",
            "abs_delta_response_turn_count"
        ])

    # --------------------------------------------------------
    # 5. Suppression déterministe uniquement
    # --------------------------------------------------------

    compact = compact.drop(
        columns=sorted(
            set(columns_to_drop)
        ),
        errors="ignore"
    )

    # Ne jamais utiliser compact.T.duplicated(),
    # car cela dépend des valeurs du test.

    compact = compact.replace(
        [np.inf, -np.inf],
        np.nan
    ).fillna(0.0)

    if len(compact) != len(feature_df):
        raise RuntimeError(
            "Le nombre de lignes a changé "
            "pendant la transformation pairwise."
        )

    return compact

In [12]:
# ============================================================
# ÉTAPE 4 — RECONSTRUIRE LES 143 FEATURES V9
# VERSION ROBUSTE POUR LE TEST CACHÉ
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. VÉRIFIER LES OBJETS NÉCESSAIRES
# ------------------------------------------------------------

required_objects = [
    "feature_namespace",
    "test_inference",
    "test_semantic_features",
    "selected_features_v9_compact",
    "build_table_safely",
    "build_compact_pairwise_deterministic",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Objets manquants avant l'Étape 4 : "
        + ", ".join(missing_objects)
    )


if len(selected_features_v9_compact) != 143:
    raise ValueError(
        "selected_features_v9_compact doit "
        "contenir exactement 143 variables."
    )


NUMBER_OF_ROWS = len(test_inference)


# ------------------------------------------------------------
# 2. RÉCUPÉRER LES FONCTIONS
# ------------------------------------------------------------

required_function_names = [
    "build_feature_table",
    "add_shorter_binary_features",
    "build_constraint_feature_table",
    "build_advanced_feature_table",
    "build_readability_feature_table",
    "build_subquestion_feature_table",
    "build_reasoning_factuality_table",
    "build_code_technical_table",
    "build_tone_safety_role_table",
]


missing_function_names = [
    name
    for name in required_function_names
    if name not in feature_namespace
]


if missing_function_names:
    raise KeyError(
        "Fonctions absentes de feature_namespace : "
        + ", ".join(missing_function_names)
    )


build_feature_table = feature_namespace[
    "build_feature_table"
]

add_shorter_binary_features = feature_namespace[
    "add_shorter_binary_features"
]

build_constraint_feature_table = feature_namespace[
    "build_constraint_feature_table"
]

build_advanced_feature_table = feature_namespace[
    "build_advanced_feature_table"
]

build_readability_feature_table = feature_namespace[
    "build_readability_feature_table"
]

build_subquestion_feature_table = feature_namespace[
    "build_subquestion_feature_table"
]

build_reasoning_factuality_table = feature_namespace[
    "build_reasoning_factuality_table"
]

build_code_technical_table = feature_namespace[
    "build_code_technical_table"
]

build_tone_safety_role_table = feature_namespace[
    "build_tone_safety_role_table"
]


print("✅ Fonctions de construction récupérées")


# ------------------------------------------------------------
# 3. PETITE FONCTION DE NORMALISATION
# ------------------------------------------------------------

def normalize_feature_table(
    table,
    table_name,
):
    if not isinstance(
        table,
        pd.DataFrame,
    ):
        raise TypeError(
            f"{table_name} n'est pas un DataFrame."
        )

    table = table.reset_index(
        drop=True
    )

    if len(table) != NUMBER_OF_ROWS:
        raise ValueError(
            f"{table_name} contient {len(table)} lignes "
            f"au lieu de {NUMBER_OF_ROWS}."
        )

    table = table.replace(
        [np.inf, -np.inf],
        np.nan,
    ).fillna(0.0)

    return table


# ------------------------------------------------------------
# 4. FEATURES TEXTUELLES DE BASE
# ------------------------------------------------------------

print("\n1/9 — Features textuelles de base")

test_features_v1 = build_table_safely(
    build_feature_table,
    test_inference,
    "Features V1 test",
)

test_features_v1 = normalize_feature_table(
    test_features_v1,
    "Features V1",
)


try:
    test_features_v1 = (
        add_shorter_binary_features(
            test_features_v1
        )
    )

except Exception as error:
    print(
        "⚠️ add_shorter_binary_features a échoué :",
        type(error).__name__,
    )

    # Recréer directement les indicateurs requis
    if (
        "a_word_count"
        in test_features_v1.columns
        and
        "b_word_count"
        in test_features_v1.columns
    ):
        test_features_v1[
            "a_is_shorter_words"
        ] = (
            test_features_v1[
                "a_word_count"
            ]
            <
            test_features_v1[
                "b_word_count"
            ]
        ).astype("int8")

        test_features_v1[
            "b_is_shorter_words"
        ] = (
            test_features_v1[
                "b_word_count"
            ]
            <
            test_features_v1[
                "a_word_count"
            ]
        ).astype("int8")

        test_features_v1[
            "same_word_count"
        ] = (
            test_features_v1[
                "a_word_count"
            ]
            ==
            test_features_v1[
                "b_word_count"
            ]
        ).astype("int8")


test_features_v1 = normalize_feature_table(
    test_features_v1,
    "Features V1 après indicateurs",
)

print(
    "Dimensions V1 :",
    test_features_v1.shape,
)


# ------------------------------------------------------------
# 5. FEATURES PAIRWISE DÉTERMINISTES
# ------------------------------------------------------------

print("\n2/9 — Features pairwise compactes")

test_features_compact = (
    build_compact_pairwise_deterministic(
        test_features_v1
    )
)

test_features_compact = normalize_feature_table(
    test_features_compact,
    "Features pairwise",
)

print(
    "Dimensions pairwise déterministes :",
    test_features_compact.shape,
)


# ------------------------------------------------------------
# 6. CONTRAINTES
# ------------------------------------------------------------

print("\n3/9 — Contraintes du prompt")

test_constraint_features = build_table_safely(
    build_constraint_feature_table,
    test_inference,
    "Contraintes test",
)

test_constraint_features = normalize_feature_table(
    test_constraint_features,
    "Contraintes",
)

print(
    "Dimensions contraintes :",
    test_constraint_features.shape,
)


# ------------------------------------------------------------
# 7. FEATURES AVANCÉES
# ------------------------------------------------------------

print("\n4/9 — Features textuelles avancées")

test_advanced_features = build_table_safely(
    build_advanced_feature_table,
    test_inference,
    "Features avancées test",
)

test_advanced_features = normalize_feature_table(
    test_advanced_features,
    "Features avancées",
)

print(
    "Dimensions avancées :",
    test_advanced_features.shape,
)


# ------------------------------------------------------------
# 8. LISIBILITÉ
# ------------------------------------------------------------

print("\n5/9 — Lisibilité")

test_readability_features = build_table_safely(
    build_readability_feature_table,
    test_inference,
    "Lisibilité test",
)

test_readability_features = normalize_feature_table(
    test_readability_features,
    "Lisibilité",
)

print(
    "Dimensions lisibilité :",
    test_readability_features.shape,
)


# ------------------------------------------------------------
# 9. SOUS-QUESTIONS
# ------------------------------------------------------------

print("\n6/9 — Sous-questions")

test_subquestion_features = build_table_safely(
    build_subquestion_feature_table,
    test_inference,
    "Sous-questions test",
)

test_subquestion_features = normalize_feature_table(
    test_subquestion_features,
    "Sous-questions",
)

print(
    "Dimensions sous-questions :",
    test_subquestion_features.shape,
)


# ------------------------------------------------------------
# 10. BLOC A
# ------------------------------------------------------------

print("\n7/9 — Bloc A : raisonnement et factualité")

test_block_a_features = build_table_safely(
    build_reasoning_factuality_table,
    test_inference,
    "Bloc A test",
)

test_block_a_features = normalize_feature_table(
    test_block_a_features,
    "Bloc A",
)

print(
    "Dimensions bloc A :",
    test_block_a_features.shape,
)


# ------------------------------------------------------------
# 11. BLOC B
# ------------------------------------------------------------

print("\n8/9 — Bloc B : code et technique")

test_block_b_features = build_table_safely(
    build_code_technical_table,
    test_inference,
    "Bloc B test",
)

test_block_b_features = normalize_feature_table(
    test_block_b_features,
    "Bloc B",
)

print(
    "Dimensions bloc B :",
    test_block_b_features.shape,
)


# ------------------------------------------------------------
# 12. BLOC C
# ------------------------------------------------------------

print("\n9/9 — Bloc C : ton, sécurité et rôle")

test_block_c_features = build_table_safely(
    build_tone_safety_role_table,
    test_inference,
    "Bloc C test",
)

test_block_c_features = normalize_feature_table(
    test_block_c_features,
    "Bloc C",
)

print(
    "Dimensions bloc C :",
    test_block_c_features.shape,
)


# ------------------------------------------------------------
# 13. VÉRIFIER LES FEATURES SÉMANTIQUES
# ------------------------------------------------------------

test_semantic_features = (
    normalize_feature_table(
        test_semantic_features,
        "Features sémantiques",
    )
)

if test_semantic_features.shape[1] != 9:
    raise ValueError(
        "Les features sémantiques doivent "
        "contenir exactement 9 colonnes."
    )


# ------------------------------------------------------------
# 14. ASSEMBLER TOUTES LES FAMILLES
# ------------------------------------------------------------

candidate_tables = [
    test_features_v1,
    test_features_compact,
    test_constraint_features,
    test_advanced_features,
    test_readability_features,
    test_semantic_features,
    test_subquestion_features,
    test_block_a_features,
    test_block_b_features,
    test_block_c_features,
]


all_test_features = pd.concat(
    candidate_tables,
    axis=1,
)


# Garder la première occurrence lorsqu’un nom apparaît
# dans plusieurs familles.
all_test_features = all_test_features.loc[
    :,
    ~all_test_features.columns.duplicated(
        keep="first"
    ),
].copy()


all_test_features = all_test_features.replace(
    [np.inf, -np.inf],
    np.nan,
).fillna(0.0)


print(
    "\nNombre total de variables candidates :",
    all_test_features.shape[1],
)


# ------------------------------------------------------------
# 15. AJOUTER LES DEUX MOYENNES
# ------------------------------------------------------------

def get_or_zero(column_name):
    if column_name in all_test_features.columns:
        return all_test_features[
            column_name
        ].astype("float32")

    print(
        "⚠️ Colonne source absente, remplacée par 0 :",
        column_name,
    )

    return pd.Series(
        np.zeros(
            NUMBER_OF_ROWS,
            dtype=np.float32,
        ),
        index=all_test_features.index,
    )


all_test_features[
    "mean_apology_count"
] = (
    get_or_zero("a_apology_count")
    + get_or_zero("b_apology_count")
) / 2.0


all_test_features[
    "mean_refusal_count"
] = (
    get_or_zero("a_refusal_count")
    + get_or_zero("b_refusal_count")
) / 2.0


print("✅ mean_apology_count ajoutée")
print("✅ mean_refusal_count ajoutée")


# ------------------------------------------------------------
# 16. AJOUTER LES FEATURES V9 ABSENTES AVEC 0
# ------------------------------------------------------------

missing_features = [
    column
    for column in selected_features_v9_compact
    if column not in all_test_features.columns
]


print(
    "Variables V9 absentes avant correction :",
    len(missing_features),
)


for column in missing_features:
    all_test_features[column] = 0.0


if missing_features:
    print(
        "⚠️ Variables ajoutées avec 0 :",
        missing_features,
    )


# ------------------------------------------------------------
# 17. CRÉER LA MATRICE FINALE
# ------------------------------------------------------------

X_test_v9_rebuilt = (
    all_test_features[
        selected_features_v9_compact
    ]
    .copy()
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
    .fillna(0.0)
    .astype("float32")
)


expected_shape = (
    NUMBER_OF_ROWS,
    143,
)


if X_test_v9_rebuilt.shape != expected_shape:
    raise ValueError(
        "Dimensions incorrectes pour X_test_v9_rebuilt : "
        f"{X_test_v9_rebuilt.shape}, "
        f"attendu {expected_shape}."
    )


if list(
    X_test_v9_rebuilt.columns
) != selected_features_v9_compact:
    raise ValueError(
        "L’ordre des 143 variables est incorrect."
    )


if not np.isfinite(
    X_test_v9_rebuilt.to_numpy()
).all():
    raise ValueError(
        "La matrice finale contient encore "
        "des valeurs non finies."
    )


print(
    "\n✅ Matrice V9 reconstruite :",
    X_test_v9_rebuilt.shape,
)

print(
    "✅ Les 143 colonnes finales sont disponibles."
)

✅ Fonctions de construction récupérées

1/9 — Features textuelles de base


Features V1 test:   0%|          | 0/3 [00:00<?, ?it/s]

Dimensions V1 : (3, 106)

2/9 — Features pairwise compactes
Dimensions pairwise déterministes : (3, 81)

3/9 — Contraintes du prompt


Contraintes test:   0%|          | 0/3 [00:00<?, ?it/s]

Dimensions contraintes : (3, 68)

4/9 — Features textuelles avancées


...

Dimensions bloc A : (3, 84)

8/9 — Bloc B : code et technique


Bloc B test:   0%|          | 0/3 [00:00<?, ?it/s]

Dimensions bloc B : (3, 150)

9/9 — Bloc C : ton, sécurité et rôle


Bloc C test:   0%|          | 0/3 [00:00<?, ?it/s]

Dimensions bloc C : (3, 156)

Nombre total de variables candidates : 748
✅ mean_apology_count ajoutée
✅ mean_refusal_count ajoutée
Variables V9 absentes avant correction : 0

✅ Matrice V9 reconstruite : (3, 143)
✅ Les 143 colonnes finales sont disponibles.


In [13]:
# ============================================================
# ÉTAPE 5 — PRÉDIRE ET CRÉER submission.csv
# VERSION ROBUSTE POUR LE TEST CACHÉ
# ============================================================

from pathlib import Path
import joblib
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. VÉRIFIER LES OBJETS DES ÉTAPES PRÉCÉDENTES
# ------------------------------------------------------------

required_objects = [
    "FINAL_MODEL_DIR",
    "metadata",
    "X_test_v9_rebuilt",
    "selected_features_v9_compact",
    "feature_namespace",
    "test_inference",
    "test_raw",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Objets manquants avant la prédiction : "
        + ", ".join(missing_objects)
    )


FINAL_MODEL_DIR = Path(
    FINAL_MODEL_DIR
)


required_model_paths = [
    FINAL_MODEL_DIR / "lightgbm_seed_2028.joblib",
    FINAL_MODEL_DIR / "lightgbm_seed_31415.joblib",
    FINAL_MODEL_DIR / "lightgbm_seed_98765.joblib",
    FINAL_MODEL_DIR / "logistic_final.joblib",
]

missing_model_paths = [
    path
    for path in required_model_paths
    if not path.is_file()
]

if missing_model_paths:
    raise FileNotFoundError(
        "Modèles manquants :\n"
        + "\n".join(
            str(path)
            for path in missing_model_paths
        )
    )


if "swap_v9_features" not in feature_namespace:
    raise KeyError(
        "swap_v9_features est absente "
        "de feature_namespace."
    )


# ------------------------------------------------------------
# 2. CONFIGURATION DU BLEND
# ------------------------------------------------------------

LGB_WEIGHT = float(
    metadata.get(
        "lgb_weight",
        0.94
    )
)

LOGISTIC_WEIGHT = float(
    metadata.get(
        "logistic_weight",
        0.06
    )
)

TEMPERATURE = float(
    metadata.get(
        "temperature",
        0.99
    )
)


if not np.isclose(
    LGB_WEIGHT + LOGISTIC_WEIGHT,
    1.0
):
    raise ValueError(
        "Les poids du blend ne totalisent pas 1."
    )

if TEMPERATURE <= 0:
    raise ValueError(
        "La température doit être positive."
    )


print("✅ Dossier des modèles :")
print(FINAL_MODEL_DIR)

print("\nConfiguration :")
print("Poids LightGBM :", LGB_WEIGHT)
print("Poids Logistic :", LOGISTIC_WEIGHT)
print("Température :", TEMPERATURE)


# ------------------------------------------------------------
# 3. CHARGER LES MODÈLES
# ------------------------------------------------------------

lgb_models = [
    joblib.load(
        FINAL_MODEL_DIR
        / "lightgbm_seed_2028.joblib"
    ),
    joblib.load(
        FINAL_MODEL_DIR
        / "lightgbm_seed_31415.joblib"
    ),
    joblib.load(
        FINAL_MODEL_DIR
        / "lightgbm_seed_98765.joblib"
    ),
]

logistic_model = joblib.load(
    FINAL_MODEL_DIR
    / "logistic_final.joblib"
)


print("\n✅ 3 modèles LightGBM chargés")
print("✅ Modèle logistique chargé")


# ------------------------------------------------------------
# 4. PRÉPARER LES FEATURES
# ------------------------------------------------------------

if len(selected_features_v9_compact) != 143:
    raise ValueError(
        "La liste finale doit contenir 143 features."
    )


missing_features = [
    column
    for column in selected_features_v9_compact
    if column not in X_test_v9_rebuilt.columns
]

if missing_features:
    raise KeyError(
        "Features absentes : "
        + ", ".join(missing_features[:20])
    )


X_original = (
    X_test_v9_rebuilt[
        selected_features_v9_compact
    ]
    .copy()
    .astype("float32")
)


if X_original.shape != (
    len(test_inference),
    143
):
    raise ValueError(
        "Dimensions incorrectes pour X_original : "
        f"{X_original.shape}"
    )


X_original = X_original.replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0.0)


swap_v9_features = feature_namespace[
    "swap_v9_features"
]


X_swapped = swap_v9_features(
    X_original
)

X_swapped = (
    X_swapped
    .reindex(
        columns=selected_features_v9_compact
    )
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0.0)
    .astype("float32")
)


if X_swapped.shape != X_original.shape:
    raise ValueError(
        "La matrice inversée a des dimensions incorrectes."
    )


print("\n✅ Matrice originale :", X_original.shape)
print("✅ Matrice inversée :", X_swapped.shape)


# ------------------------------------------------------------
# 5. FONCTIONS DE PRÉDICTION
# ------------------------------------------------------------

def align_probabilities(
    model,
    probabilities
):
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64
    )

    classes = np.asarray(
        model.classes_,
        dtype=int
    )

    aligned = np.zeros(
        (
            len(probabilities),
            3
        ),
        dtype=np.float64
    )

    for source_index, class_value in enumerate(
        classes
    ):
        if class_value not in (0, 1, 2):
            raise ValueError(
                f"Classe inattendue : {class_value}"
            )

        aligned[:, class_value] = (
            probabilities[:, source_index]
        )

    return aligned


def normalize_probabilities(
    probabilities
):
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64
    )

    probabilities = np.clip(
        probabilities,
        1e-15,
        1.0
    )

    row_sums = probabilities.sum(
        axis=1,
        keepdims=True
    )

    if np.any(row_sums <= 0):
        raise ValueError(
            "Somme de probabilités invalide."
        )

    return probabilities / row_sums


def swap_back_probabilities(
    probabilities
):
    return probabilities[:, [1, 0, 2]]


def apply_temperature(
    probabilities,
    temperature
):
    probabilities = normalize_probabilities(
        probabilities
    )

    logits = np.log(
        np.clip(
            probabilities,
            1e-15,
            1.0
        )
    )

    logits = logits / temperature

    logits = logits - logits.max(
        axis=1,
        keepdims=True
    )

    return normalize_probabilities(
        np.exp(logits)
    )


# ------------------------------------------------------------
# 6. PRÉDICTIONS LIGHTGBM
# ------------------------------------------------------------

lgb_predictions = []


for index, model in enumerate(
    lgb_models,
    start=1
):
    original_probabilities = align_probabilities(
        model,
        model.predict_proba(
            X_original
        )
    )

    swapped_probabilities = align_probabilities(
        model,
        model.predict_proba(
            X_swapped
        )
    )

    swapped_probabilities = (
        swap_back_probabilities(
            swapped_probabilities
        )
    )

    tta_probabilities = normalize_probabilities(
        (
            original_probabilities
            + swapped_probabilities
        )
        / 2.0
    )

    lgb_predictions.append(
        tta_probabilities
    )

    print(
        f"✅ LightGBM {index}/3 prédit"
    )


lgb_mean_probabilities = (
    normalize_probabilities(
        np.mean(
            lgb_predictions,
            axis=0
        )
    )
)


# ------------------------------------------------------------
# 7. PRÉDICTIONS LOGISTIQUES
# ------------------------------------------------------------

logistic_original = align_probabilities(
    logistic_model,
    logistic_model.predict_proba(
        X_original
    )
)

logistic_swapped = align_probabilities(
    logistic_model,
    logistic_model.predict_proba(
        X_swapped
    )
)

logistic_swapped = swap_back_probabilities(
    logistic_swapped
)

logistic_tta_probabilities = (
    normalize_probabilities(
        (
            logistic_original
            + logistic_swapped
        )
        / 2.0
    )
)


print("✅ Logistic prédit")


# ------------------------------------------------------------
# 8. BLEND FINAL
# ------------------------------------------------------------

final_probabilities = (
    LGB_WEIGHT
    * lgb_mean_probabilities
    + LOGISTIC_WEIGHT
    * logistic_tta_probabilities
)

final_probabilities = apply_temperature(
    final_probabilities,
    TEMPERATURE
)


if final_probabilities.shape != (
    len(test_raw),
    3
):
    raise ValueError(
        "Dimensions incorrectes pour "
        "les probabilités finales."
    )

if not np.isfinite(
    final_probabilities
).all():
    raise ValueError(
        "Probabilités finales non finies."
    )

if not np.allclose(
    final_probabilities.sum(axis=1),
    1.0
):
    raise ValueError(
        "Les probabilités ne totalisent pas 1."
    )


print("\nProbabilités finales :")

display(
    pd.DataFrame(
        final_probabilities,
        columns=[
            "winner_model_a",
            "winner_model_b",
            "winner_tie"
        ]
    ).head()
)


# ------------------------------------------------------------
# 9. CRÉER submission.csv
# ------------------------------------------------------------

submission = pd.DataFrame({
    "id":
        test_raw["id"].to_numpy(),

    "winner_model_a":
        final_probabilities[:, 0],

    "winner_model_b":
        final_probabilities[:, 1],

    "winner_tie":
        final_probabilities[:, 2],
})


probability_columns = [
    "winner_model_a",
    "winner_model_b",
    "winner_tie"
]


if len(submission) != len(test_raw):
    raise ValueError(
        "Nombre de lignes incorrect."
    )

if submission.isna().sum().sum() != 0:
    raise ValueError(
        "La soumission contient des NaN."
    )

if not np.allclose(
    submission[
        probability_columns
    ].sum(axis=1),
    1.0
):
    raise ValueError(
        "Les probabilités de la soumission "
        "ne totalisent pas 1."
    )


if np.allclose(
    submission[
        probability_columns
    ].to_numpy(),
    1.0 / 3.0
):
    raise RuntimeError(
        "Les prédictions sont uniformes à 1/3."
    )


submission_path = Path(
    "/kaggle/working/submission.csv"
)

submission.to_csv(
    submission_path,
    index=False
)


if not submission_path.is_file():
    raise FileNotFoundError(
        "submission.csv n'a pas été créé."
    )


print("\n✅ Fichier créé :")
print(submission_path)

print(
    "Dimensions :",
    submission.shape
)

display(
    submission.head()
)

print(
    "\n✅ submission.csv contient "
    "les vraies prédictions V9."
)

✅ Dossier des modèles :
/kaggle/input/datasets/davoine/donne-lui-un-nom-comme-llm-arena-final-inference/v9_final_models

Configuration :
Poids LightGBM : 0.94
Poids Logistic : 0.06
Température : 0.99

✅ 3 modèles LightGBM chargés
✅ Modèle logistique chargé

✅ Matrice originale : (3, 143)
✅ Matrice inversée : (3, 143)
✅ LightGBM 1/3 prédit
✅ LightGBM 2/3 prédit
✅ LightGBM 3/3 prédit
✅ Logistic prédit

Probabilités finales :


,winner_model_a,winner_model_b,winner_tie
0,0.305744,0.293216,0.401040
1,0.445521,0.300342,0.254138
2,0.146459,0.571912,0.281629



✅ Fichier créé :
/kaggle/working/submission.csv
Dimensions : (3, 4)


,id,winner_model_a,winner_model_b,winner_tie
0,136060,0.305744,0.293216,0.401040
1,211333,0.445521,0.300342,0.254138
2,1233961,0.146459,0.571912,0.281629



✅ submission.csv contient les vraies prédictions V9.


In [14]:
# ============================================================
# MARQUEUR FINAL DE DIAGNOSTIC
# ============================================================

from pathlib import Path

Path("/kaggle/working/final_pipeline_status.txt").write_text(
    (
        f"test_rows={len(test_raw)}\n"
        f"semantic_shape={test_semantic_features.shape}\n"
        f"features_shape={X_test_v9_rebuilt.shape}\n"
        f"submission_shape={submission.shape}\n"
        f"submission_path=/kaggle/working/submission.csv\n"
    ),
    encoding="utf-8"
)

print("✅ Marqueur final créé")

✅ Marqueur final créé
